# 03 · Interactive retrieval

Builds per-modality FAISS galleries (reloaded from the `faiss/` store on warm
starts) and runs ranked queries. Retrieval here is *semantic*: relevance = same
land-cover class, whether or not the query and gallery come from the same sensor.

In [ ]:
import os, sys
PROJ = os.path.dirname(os.getcwd()) if os.path.split(os.getcwd())[1] == 'notebooks' else os.getcwd()
if PROJ not in sys.path: sys.path.insert(0, PROJ)
nb_dir = os.path.join(PROJ, 'notebooks')
if nb_dir not in sys.path: sys.path.insert(0, nb_dir)
import utils
print('project root:', PROJ)


In [ ]:
import numpy as np
P = utils.load_pipeline('configs/default.yaml')
engine, cfg = P['engine'], P['cfg']
# Demo split: gallery = second half of the dataset, queries = first half.
n = len(P['full_ds'])
gallery_ids = np.arange(n//2, n); query_ids = np.arange(0, n//2)
galleries = {m: engine.build_gallery(gallery_ids, m) for m in cfg['modalities']}
print({m: g.size for m, g in galleries.items()})

In [ ]:
# Helper: search and show the montage.
def search(query_id, query_mod, gallery_mod, k=5):
    res = engine.retrieve(galleries[gallery_mod], [int(query_id)], query_mod, k=k)
    n_rel = int(res.relevant_mask()[0].sum())
    print(f'{query_mod}->{gallery_mod}  query #{int(query_id)}  P@{k} = {n_rel}/{k}  '
          f'{res.search_times_ms[0]:.2f} ms')
    return utils.show_retrieval_montage(P, res, k=k)
print('defined search(query_id, query_mod, gallery_mod, k=5)')

#### Same-modal retrieval

In [ ]:
search(12, 'optical', 'optical', k=5);

In [ ]:
search(12, 'sar', 'sar', k=5);

#### Cross-modal retrieval (the interesting case)

In [ ]:
search(12, 'optical', 'multispectral', k=5);

In [ ]:
search(12, 'optical', 'sar', k=5);

In [ ]:
search(500, 'sar', 'optical', k=5);

### Batch evaluation over a sample of queries
Average precision@5 for each modality pair over e.g. 100 queries.

In [ ]:
rng = np.random.RandomState(0)
qs = rng.choice(query_ids, size=100, replace=False)
pairs = [('optical','optical'), ('optical','multispectral'), ('optical','sar'),
         ('multispectral','optical'), ('sar','optical'), ('sar','sar')]
print(f'{"pair":>18}  {"P@5":>7}  {"time (ms)":>10}')
for qm, gm in pairs:
    res = engine.retrieve(galleries[gm], qs, qm, k=5)
    p5 = res.relevant_mask()[:, :5].mean()
    print(f'{qm+"->"+gm:>18}  {p5:7.3f}  {res.search_times_ms.mean():10.3f}')

### Interactive widget (optional)
If `ipywidgets` is installed, uncomment and run the cell below for a slider/
dropdown-driven demo; otherwise the parameterised `search()` above does the same.

In [ ]:
try:
    import ipywidgets as w
    from IPython.display import display
    ui = w.interactive(search, query_id=w.IntSlider(0, 0, len(P['full_ds'])-1, step=1),
                       query_mod=w.Dropdown(cfg['modalities']),
                       gallery_mod=w.Dropdown(cfg['modalities']), k=w.Dropdown([3,5,10]))
    display(ui)
except ImportError:
    print('ipywidgets not installed -- skip (pip install ipywidgets)')

---
Next: [04_faiss_index_comparison.ipynb](04_faiss_index_comparison.ipynb) compares
flat / IVF / HNSW FAISS index flavours for speed and recall.